# Setup
## Necessary library imports

In [8]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tabulate import tabulate
import numpy as np
import sqlite3
import re
import pandas as pd
import unicodedata

# Extract 
## Getting the players, player_stats, teams and plays data from those CSV files

In [11]:
#Get players
file_path = "base_data/players.csv"

players_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get players stats for different leagues

leagues = ["ENG.1","ESP.1","FRA.1","GER.1","ITA.1"]

stats_dfs = []
for league in leagues:
    file_path = f"playerStats_data/playerStats_2024_{league}.csv"
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "excel4soccer/espn-soccer-data",
        file_path,
    )
    #Putting the league code in the league column of df
    df["league"] = league
    stats_dfs.append(df)

player_stats_2024_df = pd.concat(stats_dfs, ignore_index=True) 

#Get teams
file_path = "base_data/teams.csv"

teams_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get plays for all leagues
plays_dfs = []
for league in leagues:
    file_path = f"plays_data/plays_2024_{league}.csv"
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "excel4soccer/espn-soccer-data",
        file_path,
    )
    plays_dfs.append(df)

plays_df = pd.concat(plays_dfs, ignore_index=True)

Resuming download from 9437184 bytes (2400824 bytes left)...
Resuming download to /home/wtc/.cache/kagglehub/datasets/excel4soccer/espn-soccer-data/versions/586/plays_data/plays_2024_ITA.1.csv (9437184/11838008) bytes left.


100%|██████████| 11.3M/11.3M [00:02<00:00, 864kB/s]


In [12]:
print("Stats shape:", player_stats_2024_df.shape)
print("Plays shape:", plays_df.shape)
print("Leagues:", player_stats_2024_df["league"].value_counts())

Stats shape: (3902, 21)
Plays shape: (218890, 23)
Leagues: league
ESP.1    870
ITA.1    831
ENG.1    805
FRA.1    757
GER.1    639
Name: count, dtype: int64


# Transform
## Creating a unified dataframe with combined stats, calculated metrics, and percentiles

In [14]:
# Drop columns I won't be using in each df

players_df = players_df[[
    "athleteId",
    "fullName",
    "slug",
    "weight",
    "displayHeight",
    "height",
    "age",
    "dateOfBirth",
    "citizenship",
    "positionAbbreviation"
]]

player_stats_2024_df = player_stats_2024_df[[
    "teamId",
    "athleteId",
    "league",
    "appearances_value",
    "foulsCommitted_value",
    "foulsSuffered_value",
    "yellowCards_value",
    "redCards_value",
    "ownGoals_value",
    "goalAssists_value",
    "offsides_value",
    "shotsOnTarget_value",
    "totalShots_value",
    "totalGoals_value",
    "shotsFaced_value",
    "saves_value",
    "goalsConceded_value"
]]

# Handling players who played for multiple clubs and appear as duplicates by aggregating their stats

aggregate_dict = {
    "teamId": "first", 
    "league": "first",
    "appearances_value": "sum",
    "foulsCommitted_value": "sum",
    "foulsSuffered_value": "sum",
    "yellowCards_value": "sum",
    "redCards_value": "sum",
    "ownGoals_value": "sum",
    "goalAssists_value": "sum",
    "offsides_value": "sum",
    "shotsOnTarget_value": "sum",
    "totalShots_value": "sum",
    "totalGoals_value": "sum",
    "shotsFaced_value": "sum",
    "saves_value": "sum",
    "goalsConceded_value": "sum"
}

player_stats_2024_df = (
    player_stats_2024_df
    .sort_values("appearances_value", ascending=False)
    .groupby("athleteId", as_index=False)
    .agg(aggregate_dict)
)

# Getting English first division team IDs so that I can isolate them in the teams dataframe
prem_teams_ids = player_stats_2024_df["teamId"].unique().tolist()

# Creating a dataframe tht's only English first division teams
prem_teams_df = teams_df[teams_df["teamId"].isin(prem_teams_ids)]

# Merge players.csv and player_stats.csv on athleteId
first_merged_df = players_df.merge(player_stats_2024_df, on="athleteId")

# Merge unified players dataframes with teams dataframe so that team name, colours and logos are added as columns
second_merged_df = first_merged_df.merge(
    teams_df[["teamId", "shortDisplayName", "color", "alternateColor", "logoURL"]]
    .drop_duplicates(subset=["teamId"]),
    on="teamId", how="left"
)

# Rename columns to something clearer
second_merged_df = second_merged_df.rename(columns={
    "shortDisplayName":"teamName",
    "color": "teamPrimaryColor",
    "alternateColor": "teamSecondaryColor",
    "logoURL": "teamLogo"})

# Cleaning and extracting names since assisters are only identified by text and not athleteId
def clean_name(name):
    if not isinstance(name, str):
        return None
    return unicodedata.normalize("NFKD", name)\
        .encode("ascii", "ignore")\
        .decode("utf-8")\
        .lower()\
        .strip()

def extract_assister(text):
    if not isinstance(text, str):
        return None
    
    match = re.search(r"Assisted by ([^.]+)", text)
    if not match:
        return None
    
    name = match.group(1)

    name = re.sub(r" (with|following|after).*", "", name)
    return name.strip()
    

# Parsing plays data to count chances created
def count_chances_created(plays_df):

    shot_types = [106, 117, 135, 70, 137, 173]

    df = plays_df[
        plays_df["typeId"].isin(shot_types) &
        plays_df["text"].str.contains("Assisted by", na=False)
    ].copy()

    df = df.drop_duplicates(subset=["playId"])

    # Extracting assister name    
    df["assister_name"] = df["text"].apply(extract_assister)
    df = df[df["assister_name"].notna()]
    df["clean_name"] = df["assister_name"].apply(clean_name)

    # Temp df to put names athleteIds and clean names together
    player_lookup = players_df[["athleteId", "fullName"]].drop_duplicates().copy()
    player_lookup = player_lookup.rename(columns={
    "athleteId": "assisterId"
    })
    player_lookup["clean_name"] = player_lookup["fullName"].apply(clean_name)

    # Merge with parts of plays_df
    df = df.merge(player_lookup, on="clean_name", how="left")
    print(df[["assister_name", "assisterId"]].head(10))
        
    return (
        df.dropna(subset=["assisterId"])
        .groupby("assisterId", as_index=False)
        .size()
        .rename(columns={"assisterId" : "athleteId","size": "chances_created"})        
    )

chances_created_df = count_chances_created(plays_df)

       assister_name  assisterId
0      Rodrigo Muniz    301525.0
1    Andreas Pereira    210143.0
2    Andreas Pereira    210143.0
3  Lisandro Martínez    233937.0
4        Mason Mount    231167.0
5           Casemiro    173666.0
6           Casemiro    173666.0
7        Amad Diallo         NaN
8    Andreas Pereira    210143.0
9    Andreas Pereira    210143.0


In [15]:
plays_df.columns

Index(['seasonType', 'eventId', 'playOrder', 'playId', 'typeId', 'text',
       'shortText', 'period', 'clockValue', 'clockDisplayValue', 'teamId',
       'scoringPlay', 'shootout', 'wallclock', 'goalPositionX',
       'goalPositionY', 'fieldpositionX', 'fieldPositionY', 'fieldPosition2X',
       'fieldPosition2Y', 'athleteId', 'participant', 'updateDateTime'],
      dtype='str')

In [16]:
# Adding calculated stats columns using numpy

unified_df = second_merged_df.copy()

# shot accuracy
unified_df["shot_accuracy"] = np.where (
    unified_df["totalShots_value"] > 0,
    np.round(
        unified_df["shotsOnTarget_value"] / unified_df["totalShots_value"] * 100,
        1
    ),
    np.nan
)

# conversion rate
unified_df["conversion_rate"] = np.where (
    unified_df["totalShots_value"] > 0,
    np.round(
        unified_df["totalGoals_value"] / unified_df["totalShots_value"] * 100,
        1
    ),
    np.nan
)

# on target conversion rate
unified_df["on_target_conversion_rate"] = np.where (
    unified_df["shotsOnTarget_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["shotsOnTarget_value"],
    np.nan
)

# save percentage
unified_df["save_percentage"] = np.where (
    (unified_df["saves_value"] + unified_df["goalsConceded_value"] > 0) & (unified_df["positionAbbreviation"]=="G"),
    np.round(
        unified_df["saves_value"] / (unified_df["saves_value"]+ unified_df["goalsConceded_value"]) * 100,
        1
    ),
    np.nan
)

# discipline score
unified_df["discipline_score"] = unified_df["yellowCards_value"] + (unified_df["redCards_value"] * 2)

# chances created
unified_df = unified_df.merge(
    chances_created_df,
    on="athleteId",
    how="left"
)

unified_df["chances_created"] = unified_df["chances_created"].fillna(0)

In [17]:
# Percentile calculation

# Percentile compared to players of the same position
def add_percentile_by_position(dataframe, column_name, new_column):
    for position in ["M","F","D", "G"]:
        position_match = (dataframe["positionAbbreviation"] == position)
        # choosing rows where the position is matched and grabbing column name
        # then ranking them based on percentile        
        dataframe.loc[position_match, new_column] = dataframe.loc[position_match, column_name].rank(pct=True) * 100
    return dataframe

# Percentile compared to every player
def add_percentile_global(dataframe, column_name, new_column):
    dataframe[new_column] = dataframe[column_name].rank(pct=True) * 100
    return dataframe

In [18]:
# Testing positional

test_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_percentile")
forwards = test_df[test_df["positionAbbreviation"] == "F"]
print(forwards[['fullName', 'totalGoals_value', 'goals_percentile']].sort_values('goals_percentile', ascending=False).head(10))
print()

# Testing global
test_df = add_percentile_global(test_df, "totalGoals_value", "goals_percentile")
print(test_df[["fullName", "positionAbbreviation", "totalGoals_value", "goals_percentile"]].sort_values("goals_percentile", ascending=False).head(10))


                fullName  totalGoals_value  goals_percentile
1186       Kylian Mbappé                31        100.000000
504        Mohamed Salah                29         99.888641
148   Robert Lewandowski                27         99.777283
239           Harry Kane                26         99.665924
1688       Mateo Retegui                25         99.554566
2182  Georges Mikautadze                24         99.443207
1248      Alexander Isak                23         99.331849
1475      Erling Haaland                22         99.164811
2232       Omar Marmoush                22         99.164811
1798     Mason Greenwood                21         98.775056

                fullName positionAbbreviation  totalGoals_value  \
1186       Kylian Mbappé                    F                31   
504        Mohamed Salah                    F                29   
148   Robert Lewandowski                    F                27   
239           Harry Kane                    F               

In [19]:
# Creating position specific percentiles for outfield players (for my radar charts)

unified_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_pct_pos")
unified_df = add_percentile_by_position(unified_df, "goalAssists_value", "assists_pct_pos")
unified_df = add_percentile_by_position(unified_df, "shot_accuracy", "shot_accuracy_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsSuffered_value", "fouls_suff_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsCommitted_value", "fouls_comm_pct_pos")
unified_df= add_percentile_by_position(unified_df, "chances_created", "chances_created_pct_pos")

# For forwards
unified_df = add_percentile_by_position(unified_df, 'conversion_rate', 'conversion_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'on_target_conversion_rate', 'on_target_conv_pct_pos')

# Creating global percentiles for leaderboards
unified_df = add_percentile_global(unified_df, "totalGoals_value", "goals_pct_global")
unified_df = add_percentile_global(unified_df, "goalAssists_value", "assists_pct_global")
unified_df = add_percentile_global(unified_df, "shot_accuracy", "shot_accuracy_pct_global")
unified_df = add_percentile_global(unified_df, "foulsSuffered_value", "fouls_suff_pct_global")
unified_df = add_percentile_global(unified_df, "foulsCommitted_value", "fouls_comm_pct_global")

# Inverted metrics

# GK metrics
unified_df = add_percentile_by_position(unified_df, 'goalsConceded_value', 'goals_conc_pct_pos')
unified_df['goals_conc_pct_pos'] = 100 - unified_df['goals_conc_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'save_percentage', 'save_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'saves_value', 'saves_pct_pos')

# Discipline metrics (defenders)
unified_df = add_percentile_by_position(unified_df, 'yellowCards_value', 'yellow_cards_pct_pos')
unified_df['yellow_cards_pct_pos'] = 100 - unified_df['yellow_cards_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'redCards_value', 'red_cards_pct_pos')
unified_df['red_cards_pct_pos'] = 100 - unified_df['red_cards_pct_pos']

# Offsides (forwards)
unified_df = add_percentile_by_position(unified_df, 'offsides_value', 'offsides_pct_pos')
unified_df['offsides_pct_pos'] = 100 - unified_df['offsides_pct_pos']

In [20]:
chances_created_df = count_chances_created(plays_df)


if chances_created_df["athleteId"].isna().any():
    print("[ERROR] Missing athleteId in chances_created_df")


if "chances_created" in unified_df.columns:
    unified_df = unified_df.drop(columns=["chances_created"])

unified_df = unified_df.merge(
    chances_created_df,
    on="athleteId",
    how="left"
)
unified_df["chances_created"] = unified_df["chances_created"].fillna(0).astype(int)

unified_df.sort_values("chances_created", ascending=False).head(10)

       assister_name  assisterId
0      Rodrigo Muniz    301525.0
1    Andreas Pereira    210143.0
2    Andreas Pereira    210143.0
3  Lisandro Martínez    233937.0
4        Mason Mount    231167.0
5           Casemiro    173666.0
6           Casemiro    173666.0
7        Amad Diallo         NaN
8    Andreas Pereira    210143.0
9    Andreas Pereira    210143.0


,athleteId,fullName,slug,weight,displayHeight,height,age,dateOfBirth,citizenship,positionAbbreviation,...,shot_accuracy_pct_global,fouls_suff_pct_global,fouls_comm_pct_global,goals_conc_pct_pos,save_pct_pos,saves_pct_pos,yellow_cards_pct_pos,red_cards_pct_pos,offsides_pct_pos,chances_created
2152,297373,Álex Baena,alex-baena,152.0,"5' 9""",69.0,24.0,2001-07-20T07:00Z,Spain,M,...,35.562806,95.825216,95.727804,13.644948,NaN,63.739377,4.532578,54.815864,0.755430,90
504,173896,Mohamed Salah,mohamed-salah,157.0,"5' 9""",69.0,33.0,1992-06-15T07:00Z,Egypt,F,...,83.686786,92.694127,79.418313,7.516704,NaN,28.897550,52.060134,53.619154,6.069042,86
1970,286831,Michael Olise,michael-olise,168.0,"6' 0""",72.0,24.0,2001-12-12T08:00Z,France,M,...,72.328711,99.234623,76.050654,36.402266,NaN,41.926346,38.715770,54.815864,15.250236,86
144,124091,Bruno Fernandes,bruno-fernandes,146.0,"5' 10""",70.0,31.0,1994-09-08T07:00Z,Portugal,M,...,48.715334,93.153354,88.505427,5.712937,NaN,79.320113,28.800755,0.566572,0.755430,85
2118,296395,Cole Palmer,cole-palmer,168.0,"6' 2""",74.0,23.0,2002-05-06T07:00Z,England,M,...,54.669657,98.608405,64.736989,8.026440,NaN,54.674221,10.576015,54.815864,2.171860,84
1176,231050,Raphinha,raphinha,159.0,"5' 9""",69.0,29.0,1996-12-14T08:00Z,Brazil,F,...,54.384176,91.664347,54.745338,16.146993,NaN,80.846325,14.643653,53.619154,0.167038,84
651,190161,Joshua Kimmich,joshua-kimmich,163.0,"5' 10""",70.0,31.0,1995-02-08T08:00Z,Germany,M,...,42.577488,84.010576,62.858336,27.195467,NaN,70.443815,28.800755,54.815864,4.202077,80
999,219350,Junya Ito,junya-ito,146.0,"5' 9""",69.0,32.0,1993-03-09T08:00Z,Japan,F,...,51.855628,89.743947,58.808795,6.458797,NaN,89.476615,52.060134,53.619154,9.409800,79
2162,298008,Rayan Cherki,rayan-cherki,157.0,"5' 11""",71.0,22.0,2003-08-17T07:00Z,France,M,...,75.693312,89.743947,62.858336,27.195467,NaN,17.280453,38.715770,54.815864,0.755430,72
1947,285450,Enzo Fernández,enzo-fernandez,168.0,"5' 10""",70.0,24.0,2001-01-17T08:00Z,Argentina,M,...,71.757749,97.731701,98.497078,13.644948,NaN,83.238905,7.176582,54.815864,2.927290,72


In [21]:
# Putting colours in proper hex format
unified_df["teamPrimaryColor"] = "#" + unified_df["teamPrimaryColor"]
unified_df["teamSecondaryColor"] = "#" + unified_df["teamSecondaryColor"]

# Load
## Storing the unified dataframe as a CSV file to be accessed later

In [22]:
print(unified_df.columns)

Index(['athleteId', 'fullName', 'slug', 'weight', 'displayHeight', 'height',
       'age', 'dateOfBirth', 'citizenship', 'positionAbbreviation', 'teamId',
       'league', 'appearances_value', 'foulsCommitted_value',
       'foulsSuffered_value', 'yellowCards_value', 'redCards_value',
       'ownGoals_value', 'goalAssists_value', 'offsides_value',
       'shotsOnTarget_value', 'totalShots_value', 'totalGoals_value',
       'shotsFaced_value', 'saves_value', 'goalsConceded_value', 'teamName',
       'teamPrimaryColor', 'teamSecondaryColor', 'teamLogo', 'shot_accuracy',
       'conversion_rate', 'on_target_conversion_rate', 'save_percentage',
       'discipline_score', 'goals_percentile', 'goals_pct_pos',
       'assists_pct_pos', 'shot_accuracy_pct_pos', 'fouls_suff_pct_pos',
       'fouls_comm_pct_pos', 'chances_created_pct_pos', 'conversion_pct_pos',
       'on_target_conv_pct_pos', 'goals_pct_global', 'assists_pct_global',
       'shot_accuracy_pct_global', 'fouls_suff_pct_global',
 

In [23]:
# Saving the unified df as a CSV

# unified_df.to_csv('../data/processed/unified_players_ENG_1_2024.csv', index=False)
unified_df.to_csv('../data/processed/unified_players_2024.csv', index=False)

In [24]:
# Saving it to SQLite for more efficient querying

myconnection = sqlite3.connect('../data/processed/helix_football_data.db')
# myconnection = sqlite3.connect('../data/processed/euro_helix_football_data.db')
# Saving my dataframe to a table called players in that database
unified_df.to_sql("players", myconnection, if_exists="replace", index=False)
# Closing the connection
myconnection.close()

print(f"Saved {len(unified_df)} players to helix_football_data.db")

Saved 3593 players to helix_football_data.db
